# Ch 28. 데이터 병렬 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: 4-GPU DDP 시뮬레이션에서 각 GPU가 다른 데이터로 학습한 후 그래디언트가 동일함을 보이라.

### 풀이

각 rank의 로컬 그래디언트 $g_r$를 계산한 뒤 All-Reduce 평균 $\bar g=N^{-1}\sum_r g_r$를 모든 replica에 복사한다. 따라서 동기화 직후 각 rank의 gradient tensor는 원소별로 동일하다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import torch
g=torch.Generator().manual_seed(28); w=torch.randn(3,requires_grad=True,generator=g); batches=[]; local=[]
for _ in range(4):
    x=torch.randn(5,3,generator=g); y=torch.randn(5,generator=g); batches.append((x,y))
    local.append(torch.autograd.grad(((x@w-y)**2).mean(),w)[0])
allreduce_average=torch.stack(local).mean(0)
full_x=torch.cat([x for x,_ in batches]); full_y=torch.cat([y for _,y in batches])
single_process_reference=torch.autograd.grad(((full_x@w-full_y)**2).mean(),w)[0]
max_error=float((allreduce_average-single_process_reference).abs().max())
assert torch.allclose(allreduce_average,single_process_reference,atol=1e-6,rtol=1e-6)
{'allreduce_average':allreduce_average,'full_batch_reference':single_process_reference,'max_abs_error':max_error}


## 문제 2

**문제**: 효과적 배치 크기 32, 64, 128, 256으로 학습하고 수렴을 비교하라.

### 풀이

배치만 바꾸고 총 example 수, optimizer, seed, epoch를 고정한다. train loss가 아니라 동일 validation set의 step별 loss를 비교해야 gradient noise와 update 횟수 차이를 구분할 수 있다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(2); true=np.array([2.,-1.]); X=rng.normal(size=(256,2)); y=X@true
def train(bs):
 w=np.zeros(2)
 for _ in range(20):
  for i in range(0,256,bs): w-=.1*(2/bs)*X[i:i+bs].T@(X[i:i+bs]@w-y[i:i+bs])
 return np.mean((X@w-y)**2)
loss={b:train(b) for b in [32,64,128,256]}; assert all(np.isfinite(list(loss.values()))); loss


## 문제 3

**문제**: All-Reduce의 통신량이 $O(\|\theta\|)$인 이유를 설명하라.

### 풀이

Ring All-Reduce는 크기 $P=|\theta|$인 tensor를 reduce-scatter와 all-gather에서 각각 약 $(N-1)P/N$ 전송한다. 총량은 $2(N-1)P/N<2P$이므로 rank당 $O(P)$다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
P=10_000_000; workers=[2,4,8]; volume={n:2*(n-1)/n*P for n in workers}
assert all(v<2*P for v in volume.values()); volume


## 문제 4

**문제**: 통신-연산 오버랩이 왜 속도를 높이는지 설명하라.

### 풀이

직렬 시간은 $C+M$이지만 완전 overlap의 하한은 $\max(C,M)$이다. 실제 구현은 backward bucket이 준비되는 즉시 비동기 통신을 시작해 critical path에서 일부 통신을 숨긴다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
compute,comm=8.,5.; serial=compute+comm; overlapped=max(compute,comm)
assert overlapped<serial; {'serial':serial,'overlapped_ideal':overlapped,'speedup':serial/overlapped}


## 문제 5

**문제**: LARS가 왜 큰 배치에서 도움이 되는지 설명하라.

### 풀이

큰 배치에서는 층마다 weight norm과 gradient norm의 비율이 크게 다를 수 있다. LARS는 $\eta_l=\eta\|w_l\|/(\|g_l\|+\lambda\|w_l\|)$로 층별 update 상대 크기를 제한한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
theta=np.array([100.,1.]); grad=np.array([1.,1.]); eta=.1
trust=eta*np.linalg.norm(theta)/(np.linalg.norm(grad)+1e-9)
assert trust>eta; {'global_lr':eta,'layer_scaled_lr':trust}
